# SYNAPSE — GPT-2 (decoder-only) · #10 (R1.2 / R7.4 / R7.7)

Separate notebook so GPT-2's architecture (blocks `transformer.h`, head `model.score`, no
`[CLS]` -> mean-pooled representation) is **fully decoupled** from the encoder pipeline: a bug
here cannot break the 4 encoders + GoEmotions run. It **reuses the shared, tested modules**
(`detection_metrics`, `perturbation_stats`, `ranking_agreement`, `bitflip_attack`,
`attribution_methods`) and only adds the GPT-2-specific glue. Malware task, MalwSpecSys.

Data expected at `data/GPT2/best_model_GPT2.pth` and `data/GPT2/GPT2_tokens_PT.csv`.

In [ ]:
# ==========================  GPT-2 config + cost knobs  ==========================
import os, sys, re, ast, time, random
os.environ["HF_HUB_OFFLINE"] = "1"   # usa la caché HF pre-subida (gpt2), sin internet
from collections import defaultdict
import numpy as np, pandas as pd, torch
from transformers import GPT2ForSequenceClassification
import neurox.interpretation.linear_probe as linear_probe
import detection_metrics, perturbation_stats, ranking_agreement, bitflip_attack, attribution_methods
from sklearn.metrics import f1_score

T0 = time.perf_counter()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[gpt2] device:", device)

MODEL = "GPT2"
BASE_PATH   = f"data/{MODEL}"
WEIGHTS     = f"{BASE_PATH}/best_model_{MODEL}.pth"
TOKENS_CSV  = f"{BASE_PATH}/reduced/{MODEL}_tokens_reduced.csv"  # balanced subset (2500 rows); avoids the 13GB file
NUM_LABELS  = 5
# Global label encoding = the model's class space (alphabetical LabelEncoder on the path
# labels, same convention as the malware pipeline -> Normal=2). NORMAL_IDX derived.
from sklearn.preprocessing import LabelEncoder
LABEL_ENCODER = LabelEncoder().fit(pd.read_csv(TOKENS_CSV, usecols=["label"])["label"].astype(str))
LABEL2IDX = {str(c): int(i) for i, c in enumerate(LABEL_ENCODER.classes_)}
NORMAL_IDX = next(i for c, i in LABEL2IDX.items() if "normal" in c.lower())
print("[gpt2][labels]", {os.path.basename(c): i for c, i in LABEL2IDX.items()}, "| NORMAL_IDX=", NORMAL_IDX)

# --- cost knobs (mirror the main notebook; tune before a timing run) ---
GPT2_N_SAMPLES        = 200           # balanced random sample size (CSV is huge + class-sorted)
GPT2_MAX_LEN          = 1024          # GPT-2 native context (decision: 1024; drop to 512 if too slow)
SWEEP_PCTS = [0.025, 0.05, 0.075, 0.10, 0.125, 0.15, 0.175, 0.20,
              0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.65, 0.75, 0.8, 0.95]
RANDOM_CONTROL_DRAWS  = 5
ATTRIBUTION_N_SAMPLES = 32
ATTRIBUTION_STEPS     = 20
RUN_SILENCING_SWEEP   = True
RUN_DETECTION_METRICS = True
RUN_RANDOM_CONTROL    = True
RUN_BITFLIP           = True
RUN_ATTRIBUTION       = True
os.makedirs(f"{BASE_PATH}/results", exist_ok=True)
print(f"[gpt2] N={GPT2_N_SAMPLES} MAXLEN={GPT2_MAX_LEN} pcts={len(SWEEP_PCTS)} draws={RANDOM_CONTROL_DRAWS}")

## Load GPT-2 (GPT2ForSequenceClassification + checkpoint)

In [ ]:
model = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=NUM_LABELS)
_sd = torch.load(WEIGHTS, map_location="cpu", weights_only=True)
_missing, _unexpected = model.load_state_dict(_sd, strict=False)
assert not _missing and not _unexpected, (f"state_dict mismatch: missing={_missing[:3]} unexpected={_unexpected[:3]}")
model.config.pad_token_id = model.config.eos_token_id
model.to(device).eval()
HIDDEN, NLAYERS = model.config.hidden_size, model.config.num_hidden_layers
TOTAL = HIDDEN * NLAYERS
print(f"[gpt2] loaded: hidden={HIDDEN} layers={NLAYERS} total_neurons={TOTAL} | head {tuple(model.score.weight.shape)}")

## Balanced random sample (the 13 GB CSV is class-sorted -> sample by random file offsets)

In [ ]:
_size = os.path.getsize(TOKENS_CSV)
random.seed(42)
with open(TOKENS_CSV) as f:
    f.readline()                                        # header
    _raw = [ln for ln in f if "," in ln and "[" in ln]
random.shuffle(_raw)
rows = []
for line in _raw[:GPT2_N_SAMPLES]:                      # parse only what we use
    _label_str = line.split(",", 1)[0].strip()
    if _label_str not in LABEL2IDX:
        continue
    lists = re.findall(r'\[[0-9,\s]+\]', line)
    if len(lists) < 2:
        continue
    rows.append({"label": LABEL2IDX[_label_str],
                 "input_ids": ast.literal_eval(lists[0])[:GPT2_MAX_LEN],
                 "attention_mask": ast.literal_eval(lists[1])[:GPT2_MAX_LEN]})
labels = np.array([r["label"] for r in rows], dtype=int)
import collections
print(f"[gpt2] sampled {len(rows)} rows (of {len(_raw)} in reduced CSV) | class counts:", dict(collections.Counter(labels.tolist())))

## Masked mean-pool extraction -> X, and linear probe

In [ ]:
def _hidden_meanpool(ids, mask):
    ii = torch.tensor([ids]).to(device); am = torch.tensor([mask]).to(device)
    with torch.no_grad():
        out = model(input_ids=ii, attention_mask=am, output_hidden_states=True)
    hs = torch.stack(out.hidden_states[1:]).squeeze(1)     # (L, seq, hidden)
    m = am[0].bool()
    return hs[:, m, :].mean(dim=1).reshape(-1).cpu().numpy()  # masked mean over real tokens

X = np.stack([_hidden_meanpool(r["input_ids"], r["attention_mask"]) for r in rows]).astype(np.float32)
y = labels                                                  # canonical labels (LabelEncoder, Normal=2)
_uniq = sorted(set(y.tolist())); _remap = {c: i for i, c in enumerate(_uniq)}
y_probe = np.array([_remap[c] for c in y], dtype=int)       # 0..k-1 for the probe only
probe = linear_probe.train_logistic_regression_probe(X, y_probe, lambda_l1=0.001, lambda_l2=0.001)
imp_probe = ranking_agreement.probe_importance(probe)
def top_k(pct):
    return np.argsort(imp_probe)[-max(1, round(TOTAL * pct)):].tolist()
print(f"[gpt2] X={X.shape} | probe trained | importance {imp_probe.shape}")

## GPT-2 mean-pool silencing + eval helpers

In [ ]:
# GPT-2 has no [CLS]; the representation is the mean over tokens, so silencing a neuron
# zeros it across ALL positions of the corresponding transformer block output.
def _silence_hooks(indices):
    l2i = defaultdict(list)
    for g in indices:
        l2i[int(g) // HIDDEN].append(int(g) % HIDDEN)
    handles = []
    for li, dims in l2i.items():
        dt = torch.tensor(dims, dtype=torch.long)
        def mk(dt):
            def hook(mod, inp, out):
                h = out[0] if isinstance(out, tuple) else out
                h = h.clone(); h[:, :, dt.to(h.device)] = 0.0
                return ((h,) + tuple(out[1:])) if isinstance(out, tuple) else h
            return hook
        handles.append(model.transformer.h[li].register_forward_hook(mk(dt)))
    return handles

def eval_probs(indices=()):
    hs = _silence_hooks(indices)
    try:
        P = []
        for r in rows:
            ii = torch.tensor([r["input_ids"]]).to(device); am = torch.tensor([r["attention_mask"]]).to(device)
            with torch.no_grad():
                lg = model(input_ids=ii, attention_mask=am).logits
            P.append(torch.softmax(lg, -1).squeeze(0).float().cpu().numpy())
    finally:
        for h in hs:
            h.remove()
    return np.vstack(P)

def eval_f1(indices):
    return f1_score(y, eval_probs(indices).argmax(1), average="macro", zero_division=0)

## Global silencing sweep

In [ ]:
if not RUN_SILENCING_SWEEP:
    print("[gpt2][silencing] skipped")
else:
    _t = time.perf_counter()
    base_f1 = eval_f1([])
    _rows = [{"percentage": 0.0, "macro_f1": base_f1}]
    for pct in SWEEP_PCTS:
        _rows.append({"percentage": pct, "macro_f1": eval_f1(top_k(pct))})
    pd.DataFrame(_rows).to_csv(f"{BASE_PATH}/results/global_silencing_gpt2.csv", index=False)
    print(f"[gpt2][silencing] baseline F1={base_f1:.4f} -> saved ({time.perf_counter()-_t:.1f}s)")

## Detection metrics (R3.2)

In [ ]:
if not RUN_DETECTION_METRICS:
    print("[gpt2][R3.2] skipped")
else:
    if NORMAL_IDX not in set(y.tolist()):
        print(f"[gpt2][R3.2][WARN] Normal class {NORMAL_IDX} absent in sample -> degenerate metrics")
    _rows = []
    m = detection_metrics.detection_metrics(eval_probs([]), y, normal_idx=NORMAL_IDX); m["condition"]="baseline"; _rows.append(m)
    for pct in SWEEP_PCTS:
        m = detection_metrics.detection_metrics(eval_probs(top_k(pct)), y, normal_idx=NORMAL_IDX)
        m["condition"] = f"global_silencing_{int(pct*100)}p"; _rows.append(m)
    _cols = ["condition","roc_auc","eer","tpr_at_fpr","far_argmax","mar_argmax","far_eer","mar_eer","n","n_normal","n_malicious"]
    pd.DataFrame(_rows)[_cols].to_csv(f"{BASE_PATH}/results/detection_metrics_gpt2.csv", index=False)
    print("[gpt2][R3.2] saved")

## Random-neuron control (R1.3 / R4.2)

In [ ]:
if not RUN_RANDOM_CONTROL:
    print("[gpt2][R1.3] skipped")
else:
    _base = eval_f1([]); _rows = []
    for pct in SWEEP_PCTS:
        r = perturbation_stats.top_vs_random(eval_f1, _base, top_k(pct), TOTAL,
                                             n_seeds=RANDOM_CONTROL_DRAWS, base_seed=0, higher_is_better=True)
        r["percentage"] = pct; _rows.append(r)
    _cols = ["percentage","k","baseline","top_metric","top_drop","random_mean_metric",
             "random_mean_drop","random_std_drop","p_empirical","z_score","n_seeds"]
    pd.DataFrame([{c: r[c] for c in _cols} for r in _rows]).to_csv(f"{BASE_PATH}/results/random_control_gpt2.csv", index=False)
    print("[gpt2][R1.3] saved")

## Attribution-guided bit-flip (R7.9) — on the classifier head `model.score`

In [ ]:
if not RUN_BITFLIP:
    print("[gpt2][R7.9] skipped")
else:
    _orig = model.score.weight.data.clone()
    _base = eval_f1([]); _rows = []
    for pct in SWEEP_PCTS:
        _cf = sorted({int(g) % HIDDEN for g in top_k(pct)})
        try:
            _nb = bitflip_attack.flip_exponent_msb_columns_(model.score.weight.data, _cf)
            _f1 = eval_f1([])
        finally:
            model.score.weight.data.copy_(_orig)
        _rows.append({"percentage": pct, "n_cols": len(_cf), "n_bits_flipped": _nb,
                      "baseline_f1": _base, "f1_after": _f1, "f1_drop": _base - _f1})
    pd.DataFrame(_rows).to_csv(f"{BASE_PATH}/results/bitflip_gpt2.csv", index=False)
    print("[gpt2][R7.9] saved")

## Alternative attribution rankings vs probe (#8) — mean-pool aggregation

In [ ]:
if not RUN_ATTRIBUTION:
    print("[gpt2][#8] skipped")
else:
    _layers = list(model.transformer.h)
    _sub = rows[:ATTRIBUTION_N_SAMPLES]
    def _fwd_row(r):
        ii = torch.tensor([r["input_ids"]]).to(device); am = torch.tensor([r["attention_mask"]]).to(device)
        return model(input_ids=ii, attention_mask=am).logits
    _rows = []
    imp_ag = attribution_methods.activation_times_gradient(_fwd_row, _sub, _layers, target="pred", agg="mean")
    _rows.append({"method": "activation_times_gradient", **ranking_agreement.compare_rankings(imp_probe, imp_ag, top_k_frac=0.10)})
    try:
        imp_c = attribution_methods.conductance_importance(_fwd_row, _sub, _layers, target="pred",
                                                           n_steps=ATTRIBUTION_STEPS, agg="mean")
        _rows.append({"method": "conductance", **ranking_agreement.compare_rankings(imp_probe, imp_c, top_k_frac=0.10)})
    except Exception as e:
        print(f"[gpt2][#8][WARN] conductance failed ({type(e).__name__}: {e}); act x grad only")
    pd.DataFrame(_rows).to_csv(f"{BASE_PATH}/results/attribution_agreement_gpt2.csv", index=False)
    print("[gpt2][#8] saved")

print(f"\n[gpt2] TOTAL wall-clock: {time.perf_counter()-T0:.1f}s")